In [136]:
import ast
import json
import re
import requests

import pandas as pd
import numpy as np

from bs4 import BeautifulSoup

from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 500)

url = 'https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC' # Mapa de la Comunidad de Madrid y alrededores
df = pd.read_csv('../data/pisos_madrid.csv', sep ='|')

# Extracción de datos

In [137]:
def extraer_informacion(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    estate_component = soup.find("estate-show-v2")
    if not estate_component:
        return []
    
    estate_json_raw = estate_component.get(":estate")
    estate_json_raw = estate_json_raw.replace("&quot;", '"')
    estate_data = json.loads(estate_json_raw)

    data = {
        'dormitorios': estate_data.get('rooms'),
        'superficie_m2': estate_data.get('numeric_surface'),
        'baños': estate_data.get('bathrooms'),
        'url': estate_data.get('detail_url'),
        'features': estate_data.get('features'),
        'descripcion': estate_data.get('description'),
        'precio': estate_data.get('costs'),
        'latitud': estate_data.get('latitude'),
        'longitud': estate_data.get('longitude'),
        'media': estate_data.get('media'),
        'points_of_interest': estate_data.get('points_of_interest'),
        'energy_data': estate_data.get('energy_data'),
    }
    
    return data

In [138]:
def obtener_urls(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    print(f'Buscando pisos en la página {url} ...')
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")
    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')
    estates_data = json.loads(estates_raw)

    data = []

    for estate in estates_data:
        url_piso = estate.get("detail_url")

        if url_piso in df['url'].to_list():
            print('Ya lo tengo:', url_piso)
            return data
        data.append(extraer_informacion(url_piso))

    return data

In [139]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
ultima = soup.find('a', string='>>')
max_pages = int(re.findall(r'pag-(\d+)', str(ultima))[0])

url_splited = url.split('pag-1')
data = []
for i in range(1, max_pages+1):
    subdata = obtener_urls(f'pag-{i}'.join(url_splited))
    data.extend(subdata)

    if len(subdata) < 15:
        break

df = pd.concat([pd.DataFrame(data), df])
df.to_csv('../data/pisos_madrid.csv', sep='|', index=False)

Buscando pisos en la página https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC ...
Ya lo tengo: https://www.tecnocasa.es/venta/piso/madrid/madrid/649834.html


# Pretratamiento de datos

In [140]:
X = df.drop(columns='precio')
y = df['precio'].replace(' €', '', regex=True).apply(ast.literal_eval).apply(lambda x: x.get('price')).str.replace('.', '').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [141]:
# A partir de este punto sería interesante integrar un pipeline, para que si entra un nuevo piso todo se ejecute en el orden adecuado.

In [142]:
def _safe_eval(x):
    if isinstance(x, (dict, list)):
        return x
    if pd.isna(x):
        return {}
    s = str(x).strip()
    if s in ("", "{}", "[]", "None", "nan"):
        return {}
    try:
        return ast.literal_eval(s)
    except Exception:
        return {}

def parse_nested_tecnocasa(X: pd.DataFrame) -> pd.DataFrame:
    df = X.copy()

    # 1) parseo seguro de columnas tipo dict/list que vienen como string
    for c in ["features", "precio", "media", "points_of_interest", "energy_data"]:
        if c in df.columns:
            df[c] = df[c].apply(_safe_eval)

    # 2) features
    if "features" in df.columns:
        f = df["features"]
        df["planta"] = f.apply(lambda d: d.get("floor"))
        df["aire_acondicionado"] = f.apply(lambda d: d.get("air_conditioning"))
        df["ascensor"] = f.apply(lambda d: d.get("elevator"))
        df["calefaccion"] = f.apply(lambda d: d.get("heating"))
        df["categoria"] = f.apply(lambda d: d.get("category"))
        df["año_construccion"] = f.apply(lambda d: d.get("build_year"))

    # 3) precio
    if "precio" in df.columns:
        p = df["precio"]
        df["precio_euros"] = p.apply(lambda d: d.get("price") if isinstance(d, dict) else np.nan)

        df["precio_euros"] = pd.to_numeric(p.apply(lambda d: d.get("price") if isinstance(d, dict) else np.nan)
                                           .astype(str).str.replace("€", "", regex=False).str.replace(" ", "", regex=False)
                                           .str.replace(".", "", regex=False).str.replace(",", ".", regex=False),errors="coerce" )
        df["hipoteca"] = p.apply(lambda d: d.get("mortgage_payment_value") if isinstance(d, dict) else np.nan).str.replace(".", "", regex=False).astype(int, errors="ignore")

    # 4) media
    if "media" in df.columns:
        m = df["media"]
        df["planos"] = m.apply(lambda d: d.get("floor_plans") is not None)
        df["realistico"] = m.apply(lambda d: d.get("has_realistico"))
        df["fotografias"] = m.apply(lambda d: len(d.get("images", [])) if isinstance(d.get("images", []), list) else 0)

    # 5) points_of_interest
    if "points_of_interest" in df.columns:
        poi = df["points_of_interest"]
        df["transporte_publico"] = poi.apply(lambda d: d.get("public_transport"))
        df["escuelas"] = poi.apply(lambda d: d.get("school"))
        df["farmacias"] = poi.apply(lambda d: d.get("pharmacy"))
        df["hospitales"] = poi.apply(lambda d: d.get("hospital"))
        df["supermercados"] = poi.apply(lambda d: d.get("market"))
        df["tiendas"] = poi.apply(lambda d: d.get("shop"))
        df["bares"] = poi.apply(lambda d: d.get("bar"))
        df["restaurantes"] = poi.apply(lambda d: d.get("restaurant"))

    # 6) energy_data
    if "energy_data" in df.columns:
        e = df["energy_data"]
        df["clase_energetica"] = e.apply(lambda d: d.get("class_emissions"))
        df["eficiencia_energetica"] = e.apply(lambda d: d.get("efficiency"))
        df["emisiones_energeticas"] = e.apply(lambda d: d.get("emissions"))

    return df


parse_step = FunctionTransformer(parse_nested_tecnocasa)

In [143]:
df = parse_nested_tecnocasa(df)

In [144]:
POI_MAP = {
    "transporte_publico": "tp",
    "escuelas": "esc",
    "farmacias": "fca",
    "hospitales": "hosp",
    "supermercados": "super",
    "tiendas": "tda",
    "bares": "bar",
    "restaurantes": "resto",
}

def _to_m(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.nan
    t = str(s).lower().replace(",", ".").replace(" ", "")
    if "km" in t:
        return float(t.replace("km", "")) * 1000
    if "m" in t:
        return float(t.replace("m", ""))
    return np.nan

def poi_features_tecnocasa(X: pd.DataFrame) -> pd.DataFrame:
    df = X.copy()

    for col, pref in POI_MAP.items():
        if col not in df.columns:
            continue

        df[f"{pref}_cnt"] = df[col].apply(lambda lst: len(lst) if lst else 0)

        df[f"{pref}_min_dist_m"] = df[col].apply(
            lambda lst: np.nan if not lst else min(
                _to_m(d.get("distance"))
                for d in lst
                if isinstance(d, dict) and d.get("distance")
            )
        )

    return df

poi_step = FunctionTransformer(poi_features_tecnocasa)

In [145]:
df = poi_features_tecnocasa(df)

In [146]:
def final_clean_tecnocasa(X: pd.DataFrame) -> pd.DataFrame:
    df = X.copy()

    # --- Dormitorios / baños: pasar a numérico ---
    if "dormitorios" in df.columns:
        df["dormitorios"] = (df["dormitorios"]
                             .astype(str)
                             .str.replace(r"\s*dorm\.", "", regex=True)
                             .str.strip())
        df["dormitorios"] = pd.to_numeric(df["dormitorios"], errors="coerce")

    if "baños" in df.columns:
        df["baños"] = (df["baños"]
                       .astype(str)
                       .str.replace(r"\s*baño[s]*", "", regex=True)
                       .str.strip())
        df["baños"] = pd.to_numeric(df["baños"], errors="coerce")

    # --- Planta: sacar paréntesis (ej: "3 (Exterior)" -> "3") ---
    if "planta" in df.columns:
        df["planta"] = (df["planta"]
                        .astype(str)
                        .str.replace(r"\s*\(.*\)", "", regex=True)
                        .str.strip())

    # --- Aire acondicionado: limpieza simple de texto ---
    if "aire_acondicionado" in df.columns:
        df["aire_acondicionado"] = (df["aire_acondicionado"]
                                    .astype(str)
                                    .str.replace(r"\s+.+", "", regex=True)
                                    .str.strip())

    # --- Calefacción: flags y limpieza ---
    if "calefaccion" in df.columns:
        cal = df["calefaccion"].astype(str)
        df["calefaccion_gas"] = cal.str.contains("gas", case=False, na=False)
        df["calefaccion_electrica"] = cal.str.contains("eléctrica|electrica", case=False, na=False)
        df["calefaccion"] = cal.str.replace(r"\s+.+", "", regex=True).str.strip()

    # --- Energía: limpiar comas / espacios raros ---
    if "eficiencia_energetica" in df.columns:
        s = df["eficiencia_energetica"].astype(str).str.lower().str.replace(",", ".", regex=False)
        df["eficiencia_energetica"] = pd.to_numeric(s.str.extract(r"(\d+(\.\d+)?)")[0], errors="coerce")

    if "emisiones_energeticas" in df.columns:
        df["emisiones_energeticas"] = (df["emisiones_energeticas"]
                                       .astype(str)
                                       .str.replace(",", ".", regex=False)
                                       .replace({"nan": np.nan, "None": np.nan, "": np.nan})).astype(float,errors="ignore")

    # --- Booleanos: asegurar 0/1 en ascensor, planos, etc. (si existen) ---
    for b in ["aire_acondicionado", "ascensor", "planos", "realistico", "calefaccion_gas", "calefaccion_electrica"]:
        if b in df.columns:
            df[b] = (
                df[b]
                .replace({"True": True, "False": False, "true": True, "false": False, "Sí": True, "Si": True, "No": False, "1": True, "0": False})
                .fillna(False)
                .astype(bool)
                .astype(int))

    
    df = df.replace({"None": np.nan, "nan": np.nan, "": np.nan, "NA": np.nan, "N/A": np.nan})

    return df

final_clean_step = FunctionTransformer(final_clean_tecnocasa)

In [147]:
df = final_clean_tecnocasa(df)

In [148]:
df_raw = pd.read_csv('../data/pisos_madrid.csv', sep ='|')

DROP_COLS = ['url', 'features', 'descripcion', 'precio', 'media', 'points_of_interest',
             'energy_data', 'transporte_publico', 'escuelas', 'farmacias', 'hospitales',
             'supermercados', 'tiendas', 'bares', 'restaurantes']
drop_step = FunctionTransformer(
    lambda df: df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
)

In [149]:
pipe_features = Pipeline([
    ("parse_nested", parse_step),        # crea transporte_publico, escuelas, precio_euros, etc.
    ("poi_features", poi_step),          # crea tp_cnt, tp_min_dist_m, esc_cnt, ...
    ("final_clean", final_clean_step),   # regex, flags, booleans, normalización de NaN
    ("drop",drop_step)
])

df_feat = pipe_features.fit_transform(df_raw)

In [150]:
df_feat.info()

<class 'pandas.DataFrame'>
RangeIndex: 1212 entries, 0 to 1211
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dormitorios            1099 non-null   float64
 1   superficie_m2          1212 non-null   float64
 2   baños                  1113 non-null   float64
 3   latitud                1212 non-null   float64
 4   longitud               1212 non-null   float64
 5   planta                 763 non-null    str    
 6   aire_acondicionado     1212 non-null   int64  
 7   ascensor               1212 non-null   int64  
 8   calefaccion            669 non-null    str    
 9   categoria              1212 non-null   str    
 10  año_construccion       1212 non-null   int64  
 11  precio_euros           1212 non-null   int64  
 12  hipoteca               1212 non-null   int64  
 13  planos                 1212 non-null   int64  
 14  realistico             1212 non-null   int64  
 15  fotografias    

In [151]:
df_feat['planta'].value_counts()

planta
2              153
1              143
3              123
4               87
Baja            59
Media           50
5               33
Alta            29
6               23
Planta baja     16
7               13
8                9
9                8
Ático            6
10               5
-1               3
12               3
Name: count, dtype: int64

In [152]:
# Separar en X_train, X_test, y_train , y_test
# Otro pipeline

In [153]:
def create_planta_transformer():
    
    def planta_transform(X):
        X = X.copy()
        
        numeric_vals = pd.to_numeric(X["planta"], errors="coerce")
        median = numeric_vals.median()
        p80 = np.nanpercentile(numeric_vals, 80)
        
        def transform_value(val):
            if pd.isna(val):
                return median
            
            val_str = str(val).strip().lower()
            
            if val_str in ["baja", "planta baja"]:
                return 0
            elif val_str == "media":
                return median
            elif val_str == ["alta", "ático"]:
                return p80
            else:
                try:
                    return int(val)
                except:
                    return median
        
        X["planta"] = X["planta"].apply(transform_value)
        return X

    return FunctionTransformer(planta_transform, validate=False)

In [154]:
preprocessor = ColumnTransformer(
    transformers=[
        ('planta_transform', create_planta_transformer(), ['planta']),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

pipeline = Pipeline([
    ("preprocess", preprocessor)
]).set_output(transform="pandas")

df_2 = pipeline.fit_transform(df_feat)

In [155]:
df_2.corr(numeric_only=True)

,planta,dormitorios,superficie_m2,baños,latitud,longitud,aire_acondicionado,ascensor,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,eficiencia_energetica,tp_cnt,tp_min_dist_m,esc_cnt,esc_min_dist_m,fca_cnt,fca_min_dist_m,hosp_cnt,hosp_min_dist_m,super_cnt,super_min_dist_m,tda_cnt,tda_min_dist_m,bar_cnt,bar_min_dist_m,resto_cnt,resto_min_dist_m,calefaccion_gas,calefaccion_electrica
planta,1.000000,0.160751,-0.044124,0.053056,-0.003526,0.012354,0.000125,0.137152,0.008069,0.123838,0.123854,-0.001658,-0.004181,0.023101,-0.032372,0.085537,-0.021530,0.037937,-0.041613,0.075137,-0.013949,0.046626,0.022512,0.078692,0.005299,0.072067,-0.001655,0.082124,0.022547,0.065926,0.008439,0.032892,-0.078444
dormitorios,0.160751,1.000000,0.043456,0.355283,-0.003788,0.010329,-0.033251,0.027479,0.075063,0.286259,0.286253,0.016365,0.028817,0.206015,-0.060481,-0.032001,-0.023456,-0.006186,-0.021515,-0.004626,0.020660,-0.040678,0.017067,0.003445,0.014904,0.008818,0.070432,0.018289,0.063966,0.023196,0.062674,0.163710,-0.083849
superficie_m2,-0.044124,0.043456,1.000000,0.058332,0.004368,0.034131,-0.013141,-0.008744,0.105246,0.046542,0.046558,-0.007466,0.001601,-0.034076,-0.116223,-0.101191,0.014749,-0.017311,0.037260,-0.089370,0.056855,-0.068451,-0.006102,-0.138503,0.174418,-0.108824,0.175742,-0.112601,0.164537,-0.097383,0.013652,-0.002019,-0.017742
baños,0.053056,0.355283,0.058332,1.000000,0.034257,0.033137,0.101821,0.088726,0.266952,0.478343,0.478344,0.045145,0.063850,0.274156,-0.096735,-0.144587,0.030286,-0.102563,0.095374,-0.091515,0.059561,-0.084947,-0.026482,-0.060899,0.098331,-0.100526,0.019838,-0.072438,0.139289,-0.044395,0.081164,0.104900,-0.034518
latitud,-0.003526,-0.003788,0.004368,0.034257,1.000000,0.261123,-0.080122,-0.023742,-0.037560,0.229264,0.229249,-0.117109,-0.029141,-0.026783,-0.023311,0.028192,-0.096299,0.090217,-0.091329,0.061270,-0.053935,0.078811,-0.119300,-0.037988,-0.062725,0.049416,0.027346,0.048461,-0.110490,0.041387,-0.113185,0.051220,-0.001345
longitud,0.012354,0.010329,0.034131,0.033137,0.261123,1.000000,0.002936,-0.049428,0.047242,0.029553,0.029568,-0.025183,0.152041,-0.060136,-0.062463,-0.104026,0.036690,-0.092658,0.018707,-0.029403,0.004557,-0.008615,0.048206,-0.158557,0.019251,-0.124775,0.002352,-0.121811,-0.011911,-0.137011,0.019928,0.001451,-0.042860
aire_acondicionado,0.000125,-0.033251,-0.013141,0.101821,-0.080122,0.002936,1.000000,0.114023,0.096886,0.096961,0.096947,0.057176,0.017121,0.017468,0.021318,0.037794,0.013557,0.032877,0.032004,0.014903,0.034490,0.012717,0.042417,0.062269,0.031151,0.027440,0.020084,0.032683,0.023018,0.051822,0.025889,0.173499,0.062725
ascensor,0.137152,0.027479,-0.008744,0.088726,-0.023742,-0.049428,0.114023,1.000000,0.116968,0.138502,0.138498,0.061978,0.031286,0.090603,-0.059705,0.072148,-0.019187,0.027364,0.022090,0.117922,0.031168,0.087345,-0.000359,0.089676,0.052156,0.088324,0.029536,0.101226,0.014544,0.045188,0.001035,0.150640,0.038956
año_construccion,0.008069,0.075063,0.105246,0.266952,-0.037560,0.047242,0.096886,0.116968,1.000000,-0.052843,-0.052853,0.007310,-0.036450,0.060490,-0.051932,-0.411819,0.195011,-0.197482,0.211302,-0.383488,0.297891,-0.448123,0.109797,-0.317472,0.310006,-0.389724,0.291510,-0.386213,0.345598,-0.291491,0.314351,0.093111,0.018443
precio_euros,0.123838,0.286259,0.046542,0.478343,0.229264,0.029553,0.096961,0.138502,-0.052843,1.000000,1.000000,-0.013841,0.107866,0.216178,-0.092371,0.230349,-0.132914,0.158463,-0.048926,0.218193,-0.096523,0.275847,-0.117359,0.166635,-0.109995,0.177443,-0.129437,0.181920,-0.082889,0.159310,-0.155959,0.070356,-0.119601


In [156]:
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1212 entries, 0 to 1211
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   planta                 1212 non-null   float64
 1   dormitorios            1099 non-null   float64
 2   superficie_m2          1212 non-null   float64
 3   baños                  1113 non-null   float64
 4   latitud                1212 non-null   float64
 5   longitud               1212 non-null   float64
 6   aire_acondicionado     1212 non-null   int64  
 7   ascensor               1212 non-null   int64  
 8   calefaccion            669 non-null    str    
 9   categoria              1212 non-null   str    
 10  año_construccion       1212 non-null   int64  
 11  precio_euros           1212 non-null   int64  
 12  hipoteca               1212 non-null   int64  
 13  planos                 1212 non-null   int64  
 14  realistico             1212 non-null   int64  
 15  fotografias    

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_sel),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_sel),
    ],
    remainder="drop"
)